
### Comparative Effectiveness Study - PLE

##### Research Question - Does exposure to ACE inhibitor have a different risk of experiencing acute myocardial infarction while on treatment, relative to thiazide diuretic?

##### Step 1 - Import the necessary libraries


In [ ]:
%%jupyter
# import libraries
library(rD2E)
library(Strategus)
library(dplyr)

##### Step 2 - Cohort definition loading

In [ ]:
%%jupyter
tarCohortId <- XX
compCohortId <- XX
outCohortId <- XX
cohorts_set <- c(tarCohortId, compCohortId, outCohortId)
cohortDefinitionSet <- rD2E::get_cohort_definition_set(cohorts_set)

##### Step 3 - Define the network study components

In [ ]:
%%jupyter
# Define the study period
studyStartDate <- "19000101"
studyEndDate <- "20231231"

# Target-Comparator pairs
cmTcList <- data.frame(
  targetCohortId = tarCohortId,  # ACE Inhibitor users
  targetCohortName = "ACE Inhibitor",
  comparatorCohortId = compCohortId,  # Thiazide users
  comparatorCohortName = "Thiazide Diuretic"
)

# Outcome cohort
outcomeCohortId <- outCohortId  # Acute myocardial infarction

# Time-at-risk: on-treatment
timeAtRisks <- tibble(
  label = c("On-Treatment"),
  riskWindowStart = c(0),
  startAnchor = c("cohort start"),
  riskWindowEnd = c(0),
  endAnchor = c("cohort end")
)

# Define the outcome
outcomeList <- lapply(seq_len(1), function(i) {
  CohortMethod::createOutcome(
    outcomeId = outcomeCohortId,
    outcomeOfInterest = TRUE
  )
})

# Define the T-C-O structure
targetComparatorOutcomesList <- list(
  CohortMethod::createTargetComparatorOutcomes(
    targetId = cmTcList$targetCohortId,
    comparatorId = cmTcList$comparatorCohortId,
    outcomes = outcomeList
  )
)


Create settings for modules involved in the study

In [ ]:
%%Jupyter
# Setup cohort method module
cmModuleSettingsCreator <- CohortMethodModule$new()

cmAnalysisList <- list(
  CohortMethod::createCmAnalysis(
    analysisId = 1,
    description = "On-treatment risk comparison of ACEI vs Thiazide for AMI",
    getDbCohortMethodDataArgs = CohortMethod::createGetDbCohortMethodDataArgs(
      studyStartDate = studyStartDate,
      studyEndDate = studyEndDate
    ),
    createStudyPopArgs = CohortMethod::createCreateStudyPopulationArgs(
      firstExposureOnly = TRUE,
      removeDuplicateSubjects = "keep first",
      removeSubjectsWithPriorOutcome = TRUE,
      priorOutcomeLookback = 0,
      requireTimeAtRisk = FALSE,
      riskWindowStart = timeAtRisks$riskWindowStart,
      startAnchor = timeAtRisks$startAnchor,
      riskWindowEnd = timeAtRisks$riskWindowEnd,
      endAnchor = timeAtRisks$endAnchor
    )
  )
)

cohortMethodModuleSpecifications <- cmModuleSettingsCreator$createModuleSpecifications(
  cmAnalysisList = cmAnalysisList,
  targetComparatorOutcomesList = targetComparatorOutcomesList
)

In [ ]:
%%jupyter
# Cohort Generator
cgModuleSettingsCreator <- CohortGeneratorModule$new()
cohortDefinitionShared <- cgModuleSettingsCreator$createCohortSharedResourceSpecifications(cohortDefinitionSet)
cohortGeneratorModuleSpecifications <- cgModuleSettingsCreator$createModuleSpecifications()

Create the analysis specification object - including all the modules and configurations created above

In [ ]:
%%jupyter
# Create the analysis specifications ------------------------------------------
analysisSpecifications <- createEmptyAnalysisSpecificiations() |>
  addSharedResources(cohortDefinitionShared) |>
  addModuleSpecifications(cohortGeneratorModuleSpecifications) |>
  addModuleSpecifications(cohortMethodModuleSpecifications)

##### Step 4 - Execute the Strategus study

In [ ]:
%%jupyter
study_name <- "treatment_safety_study"  # Unique study name
options <- create_options(upload_results=TRUE, study_id = study_name) # set a study_id with a unique id
options$studyId <- study_name
rD2E::run_strategus_flow(analysisSpecification = analysisSpecifications, options = options)